# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path

file_path = Path("rotisserie-chicken.html")

# Load the HTML file
with open(file_path, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [8]:
from urllib.parse import urljoin
import re

# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Usar un set para evitar duplicados
recipe_urls = set()

# Patrón de una receta real de AllRecipes
pattern = re.compile(r"https://www\.allrecipes\.com/recipe/\d+/")

for link in recipe_links:
    href = link["href"]

    # Convertir enlaces relativos en absolutos
    full_url = urljoin("https://www.allrecipes.com", href)

    # Eliminar parámetros y "/" final para normalizar
    clean_url = full_url.split("?")[0].rstrip("/")

    # Guardar únicamente recetas válidas
    if pattern.match(clean_url + "/"):
        recipe_urls.add(clean_url)

# Mostrar recetas únicas
print(f"Se encontraron {len(recipe_urls)} recetas únicas:\n")

for url in sorted(recipe_urls):
    print(url)

Se encontraron 16 recetas únicas:

https://www.allrecipes.com/recipe/14531/beer-butt-chicken
https://www.allrecipes.com/recipe/19944/drunk-chicken
https://www.allrecipes.com/recipe/214618/beer-can-chicken
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken
https://www.allrecipes.com/recipe/264278/miso-honey-chicken
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken
https://www.allrecipes.com/recipe/34957/easy-barbeque

In [22]:
import pandas as pd

corpus = []

for url in list(recipe_urls):
    title = url.rstrip("/").split("/")[-1]
    title = title.replace("-", " ").title()

    corpus.append({
        "title": title,
        "description": f"Recipe about {title}",
        "ingredients": [],
        "url": url
    })

df = pd.DataFrame(corpus)

print(df.shape)
df.head()

(16, 4)


,title,description,ingredients,url
0,Beer Can Chicken,Recipe about Beer Can Chicken,[],https://www.allrecipes.com/recipe/214618/beer-...
1,Easy Barbeque Chicken,Recipe about Easy Barbeque Chicken,[],https://www.allrecipes.com/recipe/34957/easy-b...
2,Grilled Chicken Under A Brick,Recipe about Grilled Chicken Under A Brick,[],https://www.allrecipes.com/recipe/275044/grill...
3,Buttermilk Barbecue Chicken,Recipe about Buttermilk Barbecue Chicken,[],https://www.allrecipes.com/recipe/275062/butte...
4,The Best Beer Can Chicken Ever,Recipe about The Best Beer Can Chicken Ever,[],https://www.allrecipes.com/recipe/228070/the-b...


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df["document"] = (
    df["title"].fillna("") + ". " +
    df["description"].fillna("")
)

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["document"].tolist())

print("Embeddings creados:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings creados: (16, 384)


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("NumPy:", np.__version__)
print("Scikit-learn importado correctamente")